# 02 · Preprocess

**Purpose.** Turn the raw files into one compact file per subject, containing everything training and evaluation need, so no later notebook touches the raw data again.

**Runtime.** Use a **GPU** runtime: ESPIRiT coil maps are the slow step and run much faster on GPU. The loop is resumable: if the session dies, run the notebook again and finished subjects are skipped.

**Reads.** Raw M4Raw files (from the zip cache on Drive, or Zenodo), `meta/splits.csv` from notebook 01.

**Writes to Drive.**

| Path | Content |
| --- | --- |
| `processed/<split>/<scan_id>.h5` | T2 k-space (all repetitions), coil maps, per-repetition RSS images, T1 guide images, head mask (foreground, scalp included), T1→T2 shifts |
| `meta/masks.npz` | reference copy of the undersampling masks for every R (loaders can also build them on the fly with `t1t2.masks`) |
| `t1t2_mini/` and `t1t2_mini.tar` | a small copy with the same layout (3 train, 1 val, 1 test subject) for testing a training pipeline |
| `meta/preprocess_log.csv` | one row per subject: status, shifts, size, time |

**Leakage rules built in.** Coil maps and the test-time T1 shift (`shift_acs`) use only the 12 ACS lines of the one input repetition. `shift_full` uses the fully sampled T2 and is for training pairs only.

**Week 1 checkpoint covered here:** coil maps look smooth; masks and references are saved.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/leo-oxu/eval-t1-guided-t2-rec.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/eval-t1-guided-t2-rec"
    if os.path.isdir(REPO):
        if subprocess.run(["git", "-C", REPO, "pull", "-q"]).returncode:
            print("WARNING: git pull failed (local edits?). Using the code already on this VM.")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO}/requirements.txt"], check=True)
else:
    REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)

try:   # picks up edits to t1t2/ without a restart; broken in some Colab images (Python 3.13)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as e:
    print(f"autoreload unavailable ({type(e).__name__}); restart the runtime after editing t1t2/")
import t1t2
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"t1t2 {t1t2.__version__} @ {commit}")

## 1. Settings
Keep `SMOKE` the same as in notebook 01: a smoke run reads the smoke `splits.csv` and writes under `smoke/`.

In [ ]:
SMOKE = True
DRIVE_ROOT = "/content/drive/MyDrive/t1t2"
KEEP_ZIPS_ON_DRIVE = True

N_ACS = 12                                         # fully sampled centre lines
R_VALUES = (4, 6, 8, 10)                           # net acceleration: total lines = 195 / R
RANDOM_MASK_SEEDS = (0,)
ESPIRIT = dict(kernel_width=4, thresh=0.02, crop=0.0)

MINI = {"train": 3, "val": 1, "test": 1}           # subjects per split in the mini copy
ONLY_MINI = False                                  # True: process just the mini subjects (quick hand-off)

SPLITS = ["val"] if SMOKE else ["train", "val", "test"]

In [ ]:
import shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from t1t2 import m4raw, masks
from t1t2.paths import get_paths
from t1t2.fft import ifft2c
from t1t2.align import edge_map
from t1t2.coils import coil_combine
from t1t2.preprocess import process_subject, read_subject

paths = get_paths(DRIVE_ROOT if IN_COLAB else None, smoke=SMOKE)

try:                                               # GPU for ESPIRiT if CuPy can see one
    import cupy
    DEVICE = 0 if cupy.cuda.runtime.getDeviceCount() > 0 else -1
except Exception:
    DEVICE = -1
print("ESPIRiT on", "GPU" if DEVICE >= 0 else "CPU (slow: switch to a GPU runtime for the full run)")
print(f"Drive free space: {shutil.disk_usage(paths.drive).free / 1e9:.1f} GB")

## 2. Raw data and subject list
The raw files live on the Colab disk and vanish with the session, so they are fetched again here (from the zip cache on Drive if notebook 01 saved it).

In [ ]:
for split in SPLITS:
    m4raw.fetch_split(split, paths, keep_zip_on_drive=KEEP_ZIPS_ON_DRIVE)
index = m4raw.index_files(paths.raw)

splits = pd.read_csv(paths.meta / "splits.csv", dtype={"scan_id": str})
todo = splits[splits.usable & splits.split.isin(SPLITS)].sort_values(["split", "scan_id"])
mini = pd.concat([todo[todo.split == s].head(n) for s, n in MINI.items()])   # fixed: first n per split
if ONLY_MINI:
    todo = mini
todo = todo.reset_index(drop=True)
print(f"{len(todo)} subjects to process:", todo.groupby("split").size().to_dict())
print("mini subjects:", mini.groupby("split").scan_id.apply(list).to_dict())

## 3. Undersampling masks
1D Cartesian along phase encoding. **R is the true net acceleration** (the fastMRI convention): a mask samples `round(195 / R)` lines in total, made of the 12 ACS lines plus outer lines. Only acquired lines can be sampled. Rounding to whole lines makes the realised value differ slightly from R; the table shows it.

In [ ]:
manifest = pd.read_csv(paths.meta / "manifest.csv", dtype={"scan_id": str})
t2m = manifest[manifest.contrast == "T2"]
pat = t2m.groupby(["n_pe", "pe_first", "pe_last", "pe_contiguous", "dc_line"]).size()
assert len(pat) == 1, f"T2 files do not share one PE pattern:\n{pat}"
n_pe, pe_first, pe_last, contiguous, dc_line = pat.index[0]
assert contiguous and dc_line == n_pe // 2, "unexpected PE layout: revisit t1t2/masks.py"

pe_acquired = np.zeros(n_pe, bool)
pe_acquired[pe_first:pe_last + 1] = True
all_masks = masks.make_all_masks(pe_acquired, R_VALUES, N_ACS, RANDOM_MASK_SEEDS)
np.savez(paths.meta / "masks.npz", **all_masks)

names = [k for k in all_masks if k not in ("acquired", "acs")]
fig, ax = plt.subplots(figsize=(11, 0.32 * len(all_masks) + 0.8))
ax.imshow(np.stack(list(all_masks.values())), aspect="auto", cmap="gray", interpolation="nearest")
ax.set_yticks(range(len(all_masks)), list(all_masks)); ax.set_xlabel("PE line")
plt.tight_layout()
pd.DataFrame([dict(mask=k, lines=int(all_masks[k].sum()),
                   acs=N_ACS, outer=int(all_masks[k].sum()) - N_ACS,
                   net_R=round(masks.net_acceleration(all_masks[k], pe_acquired), 2)) for k in names])

## 4. One subject first: quality check and cost estimate
Process a single subject, look at the result, and estimate the time and Drive space for the full run before starting it.

In [ ]:
first = todo.iloc[0]
first_path = paths.processed / first.split / f"{first.scan_id}.h5"
row = process_subject(m4raw.subject_files(index, first.scan_id), first_path, first.scan_id,
                      first.split, n_acs=N_ACS, device=DEVICE, espirit_kwargs=ESPIRIT,
                      overwrite=True)
total_reps = int((todo.split.map(lambda x: 6 if x == "test" else 3)).sum())   # test has 6 T2 reps
per_rep_s, per_rep_mb = row["seconds"] / row["n_t2_reps"], row["size_mb"] / row["n_t2_reps"]
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
print(f"\nEstimated full run: {total_reps * per_rep_s / 3600:.1f} h, "
      f"{total_reps * per_rep_mb / 1e3:.1f} GB on Drive")

In [ ]:
d = read_subject(first_path, rep=0, aligned="full")
s = d["kspace"].shape[0] // 2
maps, bm = d["maps"][s], d["head_mask"][s]
coil_imgs = ifft2c(d["kspace"][s])
combined = np.abs(coil_combine(coil_imgs, maps))
rss_img = d["t2_rss"][0, s]

fig, ax = plt.subplots(2, maps.shape[0], figsize=(3.2 * maps.shape[0], 6.4))
for c in range(maps.shape[0]):
    ax[0, c].imshow(np.abs(maps[c]) * bm, cmap="viridis", vmin=0, vmax=1); ax[0, c].set_title(f"|map| coil {c}")
    ax[1, c].imshow(np.angle(maps[c]) * bm, cmap="twilight", vmin=-np.pi, vmax=np.pi); ax[1, c].set_title(f"phase coil {c}")
for a_ in ax.ravel(): a_.axis("off")
plt.suptitle("ESPIRiT maps from the ACS only (shown inside the head mask)"); plt.tight_layout()

err = np.linalg.norm((combined - rss_img)[bm]) / np.linalg.norm(rss_img[bm])
print(f"map-combined image vs RSS inside the head: relative difference {err:.3f}")

**Reading this.** The maps should be smooth, each coil bright on its own side of the head, with no speckle inside the head. The printed number says how much signal the ACS-only maps fail to explain; a few percent is expected at this SNR. If it is large (say above 0.1), revisit `kernel_width` or `N_ACS` before the full run.

In [ ]:
def overlay(a, b):   # red = T2 edges, green = T1 edges, yellow = they coincide
    return np.clip(np.dstack([edge_map(a), edge_map(b), np.zeros_like(a)]) * 3, 0, 1)

raw_t1 = read_subject(first_path, rep=0, slices=[s], aligned=None)["t1"][0]
acs_t1 = read_subject(first_path, rep=0, slices=[s], aligned="acs")
t2_ref = d["t2_rss"][:, s].mean(0)

fig, ax = plt.subplots(1, 5, figsize=(20, 4.4))
ax[0].imshow(t2_ref, cmap="gray"); ax[0].set_title("T2 reference (mean of reps)")
ax[1].imshow(t2_ref, cmap="gray"); ax[1].contour(bm, [0.5], colors="c", linewidths=0.8); ax[1].set_title("head mask")
ax[2].imshow(overlay(t2_ref, raw_t1)); ax[2].set_title("edges: T1 unaligned")
ax[3].imshow(overlay(t2_ref, d["t1"][s])); ax[3].set_title(f"full alignment {d['shift'].round(2)}")
ax[4].imshow(overlay(t2_ref, acs_t1["t1"][0])); ax[4].set_title(f"ACS-only alignment {acs_t1['shift'].round(2)}")
for a_ in ax: a_.axis("off")
plt.tight_layout()

## 5. All subjects
Resumable: finished subjects are skipped. A subject that fails is logged and the loop carries on.

In [ ]:
log_path = paths.meta / "preprocess_log.csv"
log = pd.read_csv(log_path, dtype={"scan_id": str}).to_dict("records") if log_path.exists() else []
log = [r for r in log if r["scan_id"] != first.scan_id] + [row]

for r in tqdm(todo.itertuples(), total=len(todo)):
    out = paths.processed / r.split / f"{r.scan_id}.h5"
    if out.exists():
        continue
    try:
        res = process_subject(m4raw.subject_files(index, r.scan_id), out, r.scan_id, r.split,
                              n_acs=N_ACS, device=DEVICE, espirit_kwargs=ESPIRIT)
    except Exception as e:
        res = dict(scan_id=r.scan_id, split=r.split, status=f"FAILED: {type(e).__name__}: {e}")
        print(res)
    log = [x for x in log if x["scan_id"] != r.scan_id] + [res]
    pd.DataFrame(log).to_csv(log_path, index=False)
pd.DataFrame(log).to_csv(log_path, index=False)

## 6. Summary

In [ ]:
log_df = pd.read_csv(log_path, dtype={"scan_id": str})
on_disk = {p.stem for p in paths.processed.rglob("*.h5")}
missing = sorted(set(todo.scan_id) - on_disk)
total_gb = sum(p.stat().st_size for p in paths.processed.rglob("*.h5")) / 1e9
print(f"processed {len(on_disk & set(todo.scan_id))}/{len(todo)} subjects, {total_gb:.1f} GB on Drive")
print("missing:", missing if missing else "none")
display(log_df[log_df.status.str.startswith("FAILED")])

done = log_df[log_df.status == "done"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for a_, axis in zip(ax, ("fe", "pe")):
    a_.scatter(done[f"shift_full_{axis}"], done[f"shift_acs_{axis}"], s=12)
    lim = [done[[f"shift_full_{axis}", f"shift_acs_{axis}"]].min().min() - 0.5,
           done[[f"shift_full_{axis}", f"shift_acs_{axis}"]].max().max() + 0.5]
    a_.plot(lim, lim, "k--", lw=0.8)
    a_.set(xlabel="shift from full T2 (px)", ylabel="shift from ACS only (px)",
           title=f"T1→T2 shift along {axis.upper()}")
plt.tight_layout()
done[["shift_full_fe", "shift_acs_fe", "shift_full_pe", "shift_acs_pe"]].describe().loc[["mean", "std", "min", "max"]]

**Reading this.** Points on the dashed line mean the realistic ACS-only alignment agrees with the full one. Along phase encoding the ACS image has very low resolution, so more scatter is expected there than along frequency encoding.

## 7. Export the mini copy
`t1t2_mini/` has exactly the same layout and file format as the full data, with only the mini subjects, so a pipeline that runs on it runs on the full data by changing one path. It is also packed into `t1t2_mini.tar` (about 1 GB), which is quick to copy to the Colab disk or download to a laptop.

In [ ]:
import tarfile

mini_root = paths.out / "t1t2_mini"
if mini_root.exists():
    shutil.rmtree(mini_root)                       # rebuilt from scratch every time
(mini_root / "meta").mkdir(parents=True)
for r in mini.itertuples():
    src = paths.processed / r.split / f"{r.scan_id}.h5"
    assert src.exists(), f"{r.scan_id} is not processed yet: run section 5 first"
    (mini_root / "processed" / r.split).mkdir(parents=True, exist_ok=True)
    shutil.copy(src, mini_root / "processed" / r.split / src.name)
mini.to_csv(mini_root / "meta" / "splits.csv", index=False)
for name in ("masks.npz", "audit_summary.json"):
    if (paths.meta / name).exists():
        shutil.copy(paths.meta / name, mini_root / "meta" / name)

tar_local = paths.local / "t1t2_mini.tar"          # build on the local disk, then copy once
with tarfile.open(tar_local, "w") as tar:
    tar.add(mini_root, arcname="t1t2_mini")
shutil.copy(tar_local, paths.out / "t1t2_mini.tar")
print(f"{paths.out / 't1t2_mini.tar'}: {tar_local.stat().st_size / 1e9:.2f} GB, "
      f"{len(mini)} subjects {mini.groupby('split').size().to_dict()}")

### Cell for whoever trains on this data
Paste this at the top of a training notebook. `USE_MINI` is the only line that changes between the mini copy and the full data. The file format is described in `t1t2/README.md`.

```python
USE_MINI = True                             # False = full data

import os, shutil, tarfile
from google.colab import drive
drive.mount("/content/drive")
SHARED = "/content/drive/MyDrive/t1t2"      # the shared project folder (or your shortcut to it)

if USE_MINI:
    if not os.path.isdir("/content/t1t2_mini"):
        shutil.copy(f"{SHARED}/t1t2_mini.tar", "/content/t1t2_mini.tar")
        with tarfile.open("/content/t1t2_mini.tar") as tar:
            tar.extractall("/content")
    DATA_ROOT = "/content/t1t2_mini"
else:
    DATA_ROOT = SHARED
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(f"{DATA_ROOT}/processed")))
```

## Checkpoint
Before moving to `03_classical_baselines`, confirm:

- [ ] Section 3: realised net R table copied into the project plan
- [ ] Section 4: coil maps are smooth; map-combined vs RSS difference is small
- [ ] Section 4: both alignment overlays are more yellow than the unaligned one
- [ ] Section 6: every subject processed, none failed
- [ ] Section 7: `t1t2_mini.tar` exists and the training side can load it

Once all subjects are processed and you are short of Drive space, `raw_zips/` can be deleted: no later notebook reads the raw data.